## Here we will learn about MCP

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient(
    {
      "time": {
          "transport": "stdio",
          "command": "uvx",
          "args": [
            "mcp-server-time"
          ]
        }
    }
)

In [ ]:
# Now fetch the tools from the client we have created
tools = await mcp_client.get_tools()

In [ ]:
from pprint import pprint
resources = await mcp_client.get_resources("time")

async with mcp_client.session("time") as session:
    prompts = await session.list_prompts()

tools = await mcp_client.get_tools()

pprint({
    "resources": resources,
    "prompts": prompts,
    "tools": tools
})


In [ ]:
from pprint import pprint
pprint(tools)

In [ ]:
## Now add the tools to the agent

from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

chat_model = ChatOpenAI(
    model="local-llm",
    base_url="http://localhost:8080/v1",
    temperature=0.7
)

agent = create_agent(
    model = chat_model,
    tools=tools
)


In [ ]:
from langchain.messages import HumanMessage

response = await agent.ainvoke({
    "messages": [HumanMessage(content="what is current time in india?")]
})


In [ ]:
from pprint import pprint
pprint(response["messages"])

# Let's use our local or custom MCP server


In [7]:
## Now lets connect to our local MCP server which can be found at 'module-2/mcp_server/server.py'
from langchain_mcp_adapters.client import MultiServerMCPClient

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.messages import HumanMessage

chat_model = ChatOpenAI(
    model="local-llm",
    base_url="http://localhost:8080/v1",
    temperature=0.7
)

everest_mcp_client = MultiServerMCPClient(
    {
        'everest_mcp_server': {
            "transport": "streamable_http",
            "url": "http://127.0.0.1:8000/mcp"
        }
    }
)

## get tools
tools = await everest_mcp_client.get_tools()
prompt = await everest_mcp_client.get_prompt('everest_mcp_server', 'prompt')
resources = await everest_mcp_client.get_resources("everest_mcp_server")

In [8]:
from pprint import pprint
pprint({
    "resources": resources,
    "prompts": prompt,
    "tools": tools
})

{'prompts': [HumanMessage(content='\n    You are a helpful assistant that answers user questions about .\n\n    You can use the following tools/resources to answer user questions:\n    - search_web: Search the web for information\n    - github_file: Access the scribble assignment repo files\n\n    If the user asks a question that is not related to Everest github, Everest scribble assignment or complete file structure in that repository, you should say "I\'m sorry, I can only answer questions about Everest scribble assignment."\n\n    You may try multiple tool and resource calls to answer the user\'s question.\n\n    You may also ask clarifying questions to the user to better understand their question.\n    ', additional_kwargs={}, response_metadata={})],
 'resources': [Blob 4625981056],
 'tools': [StructuredTool(name='search_web', description='Search the web for information', args_schema={'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'se

In [9]:
agent = create_agent(
    model = chat_model,
    tools=tools,
    system_prompt=prompt[0].content
)

response = await agent.ainvoke({
    "messages": [HumanMessage(content="What is your role?")]
})

In [10]:
from pprint import pprint

# pprint(tools)
pprint(response["messages"])

[HumanMessage(content='What is your role?', additional_kwargs={}, response_metadata={}, id='ebb04d1c-23dc-46db-b892-4b2e5d7b90aa'),
 AIMessage(content='I am a helpful assistant. My role is to answer questions specifically about the Everest scribble assignment, including its contents, file structure, or related information, utilizing the resources available to me.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 346, 'prompt_tokens': 195, 'total_tokens': 541, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': './models/gemma4-e4b-it-qat.gguf', 'system_fingerprint': 'b11146-7fe450e19', 'id': 'chatcmpl-UxXvBXS7wXikQ45S1f5gPabibTMLjugO', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a15cb7-fe7a-77e0-bc4c-8344e39198ab-0', tool_calls=[], invalid_tool_calls=[], usage_meta

In [ ]:
response = await agent.ainvoke({
    "messages": [HumanMessage(content="What is everest scribble assignment is about?")]
})

pprint(response["messages"][1].tool_calls)

[HumanMessage(content='What is everest scribble assignment is about?', additional_kwargs={}, response_metadata={}, id='8b2b022a-4f58-4f92-ac32-04df597be62c'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 129, 'prompt_tokens': 199, 'total_tokens': 328, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 193, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': './models/gemma4-e4b-it-qat.gguf', 'system_fingerprint': 'b11146-7fe450e19', 'id': 'chatcmpl-RjtMx0g6wwFSpfhPuw4fh9G3kyHOZTuV', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a15cbb-c075-7411-8292-d8faf08e735a-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'Everest scribble assignment'}, 'id': 'IwceaMCwKloD4jXDWYFDLgKvYGpEh8PO', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 199, 'output_tokens':

In [14]:
pprint(response["messages"][1].tool_calls)

[{'args': {'query': 'Everest scribble assignment'},
  'id': 'IwceaMCwKloD4jXDWYFDLgKvYGpEh8PO',
  'name': 'search_web',
  'type': 'tool_call'}]


In [15]:
response = await agent.ainvoke({
    "messages": [HumanMessage(content="what tech stack the scribble assignment is using?")]
})

In [16]:
pprint(response["messages"])

[HumanMessage(content='what tech stack the scribble assignment is using?', additional_kwargs={}, response_metadata={}, id='239b1ff0-aa89-42e1-a691-64ea7e9cc9df'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 225, 'prompt_tokens': 199, 'total_tokens': 424, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 181, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': './models/gemma4-e4b-it-qat.gguf', 'system_fingerprint': 'b11146-7fe450e19', 'id': 'chatcmpl-Xi0lGMJm0I63Ds8TvJkH4xE04Wxgt7k0', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a15cbd-7bce-7813-a341-2289868147b2-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'Everest scribble assignment repo files'}, 'id': 'mxpXSWnvpJYw4xRhoFSINqRmT44oS0J5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 199, '

In [ ]:
## Lets check What are all "Time" MCP server is providing and also Everest MCP server is providing
from pprint import pprint

print("## ===== Time MCP server ===== ##")
resources = await mcp_client.get_resources("time")

async with mcp_client.session("time") as session:
    prompts = await session.list_prompts()

tools = await mcp_client.get_tools()

pprint({
    "resources": resources,
    "prompts": prompts,
    "tools": tools
})
print("## ===== Everest MCP server ===== ##")

resources = await everest_mcp_client.get_resources("everest_mcp_server")

async with everest_mcp_client.session("everest_mcp_server") as session:
    prompts = await session.list_prompts()

tools = await everest_mcp_client.get_tools()

pprint({
    "resources": resources,
    "prompts": prompts,
    "tools": tools
})